# Group 3

In [1]:
!pip install ftfy regex tqdm torchmetrics git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-vj2ymla_
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-vj2ymla_
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 40.4 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=c521e434e6ced859a7102ab6e4e696aed16befdaf32a0709ab751a0dba426ea6
  Stored in directory: /tmp/pip-ephem-wheel-cache-stg10fhu/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [2]:
import os

# Define paths
zip_path = './aml-group-lab.zip'
extract_to = './AMLdataset'

# Automated Unzip
if not os.path.exists(extract_to):
    os.makedirs(extract_to)
    print(f"Unzipping {zip_path}...")

    # Using the shell command is usually faster for large zips
    !unzip -q {zip_path} -d {extract_to}

    print("Unzip complete!")
else:
    print("Dataset already unzipped.")

Unzipping ./aml-group-lab.zip...
Unzip complete!


In [10]:
import os
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset


class AMLDataset(Dataset):
    def __init__(self, train_csv_path, imgs_dir, train=True, transform=None):
        self.df = pd.read_csv(train_csv_path)
        self.imgs_dir = imgs_dir
        self.train = train
        self.transform = transform

        if self.train:
            # Create the 'classes' attribute (Unique list of names)
            # We sort them to ensure the index mapping is always consistent
            self.classes = sorted(self.df['label'].unique().tolist())

            # Create a mapping from Name -> Integer ID
            self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

            # Create the 'labels' attribute (The ID for every single row)
            # This is helpful if you want to use a Weighted or Balanced Sampler later
            self.labels = [self.class_to_idx[name] for name in self.df['label']]
        else:
            # Group the rows by episode_id so __len__ returns total number of tasks
            self.episode_ids = sorted(self.df['episode_id'].unique())

    def __len__(self):
        if self.train:
            return len(self.df)
        else:
            return len(self.episode_ids)

    def __getitem__(self, idx):
        if self.train:
            # Standard single image return
            row = self.df.iloc[idx]
            return self.load_img(row["filename"]), self.labels[idx]

        else:
            # Return a full 5-way 5-shot Episode
            ep_id = self.episode_ids[idx]
            ep_df = self.df[self.df['episode_id'] == ep_id]

            support_df = ep_df[ep_df['role'] == 'support']
            query_df = ep_df[ep_df['role'] == 'query']

            # Pack support images and their labels
            s_imgs = torch.stack([self.load_img(f) for f in support_df['filename']])
            s_labels = torch.tensor(support_df['label'].values)

            # Pack query images
            q_imgs = torch.stack([self.load_img(f) for f in query_df['filename']])

            # If your test CSV has query labels (sometimes they are hidden), include them:
            # q_labels = torch.tensor(query_df['label'].values) if 'label' in query_df.columns else None

            return {
                "support_imgs": s_imgs,
                "support_labels": s_labels,
                "query_imgs": q_imgs,
                "episode_id": ep_id
            }

In [ ]:
import os
import clip
import torch
import torchmetrics
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

# Define constants
LR = 1e-6 # Learning Rate
BATCH_SIZE = 32 # Number of images processed in one iteration
SHOTS = 5 # 5-way 5-shot
EPOCHS = 5 # Number of epochs

# Define controls
best_acc = 0.0 # Used to save best checkpoints

# Load the models
device = "cuda" if torch.cuda.is_available() else "cpu"
student, preprocess = clip.load('ViT-B/32', device, jit=False)
teacher, _ = clip.load('ViT-B/32', device)
for param in teacher.parameters():
    param.requires_grad = False

# Download the dataset
train_dataset = AMLDataset(
    train_csv_path="./AMLdataset/release/train.csv",
    imgs_dir="./AMLdataset/release/images/",
    train=True,
    transform=preprocess
)

test_dataset = AMLDataset(
    train_csv_path="./AMLdataset/release/test_episodes_release.csv",
    imgs_dir="./AMLdataset/release/images/",
    train=False,
    transform=preprocess
)

# Prepare the data
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_train = len(train_dataloader)

all_class_texts = torch.cat([clip.tokenize(f"a photo of a {c}") for c in train_dataset.classes]).to(device)
NUM_CLASSES = len(train_dataset.classes)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=True,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_test = len(test_dataloader)

# Prepare criterion and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.1)

# Prepare the training
for epoch in range(EPOCHS):
    print(f"Epoch: {epoch}")
    epoch_train_loss = 0
    student.train() # Set the model to training mode (enables dropout/batchnorm updates)

    print("Running Training...")
    for batch in tqdm(train_dataloader, total=num_batches_train):
        optimizer.zero_grad() # Clear previous gradients before starting a new optimization step

        # Unpack images and class indices
        # images shape: [Batch, 3, 224, 224]
        # labels shape: [Batch]
        images, class_ids = batch
        images = images.to(device)

        # Map class IDs to text descriptions
        # We clean class names (replace underscores with spaces) for better CLIP performance
        texts = [f"a photo of a {str(train_dataset.classes[i]).replace('_', ' ')}" for i in class_ids]
        texts = clip.tokenize(texts).to(device)

        # Forward pass
        # CLIP computes similarity between all images and all texts in the batch
        logits_per_image, logits_per_text = student(images, texts)

        # Define Ground Truth
        # Creates a diagonal target [0, 1, ..., N-1] where image i matches text i
        ground_truth = torch.arange(len(images), dtype=torch.long, device=device)

        # Compute Symmetric Loss
        loss_i = criterion(logits_per_image, ground_truth)
        loss_t = criterion(logits_per_text, ground_truth)
        total_loss = (loss_i + loss_t) / 2

        # Optimization
        total_loss.backward() # Perform backpropagation to calculate gradients
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0) # This helps stability
        optimizer.step() # Update model weights based on calculated gradients

        epoch_train_loss += total_loss.item()

    print(f"Epoch: {epoch} | Training Loss: {epoch_train_loss:.4f}")

Epoch: 0
Running Training...


 53%|█████▎    | 82/156 [40:55<36:21, 29.47s/it]

In [ ]:
def evaluate_episodic(model, dataloader, device):
    model.eval()
    acc_list = []

    print("Running Episodic Evaluation...")
    with torch.no_grad():
        for episode in tqdm(dataloader):
            # Unpack Episode (Dataloader batch_size must be 1)
            # Shapes: [25, 3, 224, 224]
            s_imgs = episode["support_imgs"].squeeze(0).to(device)
            s_labels = episode["support_labels"].squeeze(0).to(device)
            q_imgs = episode["query_imgs"].squeeze(0).to(device)

            # Note: If your test CSV doesn't have query labels,
            # you can't calculate accuracy here.
            # Assuming query_labels exist or query order matches support order:
            # Check if your CSV has query labels. If not, this part needs adjustment.
            # Usually, in these datasets, query labels match the support labels sequence.
            q_labels = episode.get("query_labels", None)
            if q_labels is not None:
                q_labels = q_labels.squeeze(0).to(device)

            # Extract Features
            s_features = model.encode_image(s_imgs)
            q_features = model.encode_image(q_imgs)

            # L2 Normalize (Cosine Similarity)
            s_features /= s_features.norm(dim=-1, keepdim=True)
            q_features /= q_features.norm(dim=-1, keepdim=True)

            # Compute Prototypes (Mean of 5 images per class)
            unique_labels = torch.unique(s_labels)
            prototypes = []
            for label in unique_labels:
                # Average the 5 images belonging to this class
                p = s_features[s_labels == label].mean(dim=0)
                prototypes.append(p / p.norm())
            prototypes = torch.stack(prototypes) # Shape: [5, 512]

            # Classify Queries against Prototypes
            # [25, 512] @ [512, 5] -> [25, 5]
            logits = 100.0 * q_features @ prototypes.T
            preds = logits.argmax(dim=-1)

            # Accuracy Calculation (Map preds back to original labels if needed)
            # In a 5-way setting, labels are often relative (0-4)
            # For simplicity, we compare if the predicted prototype index
            # matches the expected class relative index.
            if q_labels is not None:
                correct = (preds == q_labels).float().mean()
                acc_list.append(correct)

    if acc_list:
        mean_acc = torch.stack(acc_list).mean().item()
        return mean_acc
    else:
        print("No Query Labels found in CSV. Accuracy cannot be calculated.")
        return 0.0

evaluate_episodic(student, test_dataloader, device)